# 32.02 Условный выбор пары при известной $h$

> **Статус:** канонический синтетический перебор пар для полной разработочной
> постановки, где эффективная $h$ задана внешним КТ-референсом. Реальная пара
> эксперимента 2 не выбрана: калибровка, ковариация и принятые данные пока не
> переданы в этот расчёт.

Notebook не читает медицинские данные. Он формализует кандидатов и доказывает,
что «хорошая структурная разделимость» и «большой дыхательный сигнал» могут
предпочитать разные пары.


## Область действия и субъектные наборы

Расчёт относится только к боковым сборкам эксперимента 2 и
реокардиомонитору МГТУ. РЕО32 эксперимента 1 и реокардиомонитор РНЦХ
эксперимента 3 не используют этот результат автоматически.

- Для добровольца с поздней копией файла 100 мм независимый набор содержит
  9 размеров: 50, 60, 70, 80, 90, 110, 120, 130, 140 мм, то есть 36 пар.
- Для второго добровольца 90 и 100 мм являются разными записями; до
  окончательного QC доступны 10 размеров и 45 пар.
- Размеры записаны в одном убывающем порядке 140→50 мм и не одновременно.
  Поэтому каждая пара является псевдоодновременной и требует допущения
  воспроизводимости состояния между двумя записями.

Субъектная $h$ должна приходить из КТ по каноническому правилу сведения
реальной анатомии к эффективной толщине модели. Ниже вместо неё используется
только синтетическая точка $h=20$ мм.


## Что вычисляется для каждой пары

При фиксированной $h$ и калиброванном знаковом наблюдении рассматривается
логарифмический якобиан

$$
\widetilde J_{ij}=\begin{bmatrix}
S_{\rho_1}(L_i)&S_{\rho_2}(L_i)\\
S_{\rho_1}(L_j)&S_{\rho_2}(L_j)
\end{bmatrix}.
$$

В таблицу входят независимые показатели:

- $\sigma_{min}(\widetilde J)$ и угол строк — локальная структурная
  разделимость двух сопротивлений;
- $\kappa(\widetilde J)$ — усиление худшего относительного направления при
  данной параметризации;
- $[(\widetilde J^T\widetilde J)^{-1}]_{22}$ — **условная единично-взвешенная**
  нижняя граница дисперсии $\ln\rho_2$. Здесь $\Sigma=I$ — только нормировка,
  а не модель экспериментальной ошибки;
- минимум $|\Delta Z_{breath}|$ из двух размеров — синтетическая измеримость
  обеих строк пары, не signal-to-noise ratio.

Ни один показатель не является достаточным. Пара соседних крупных размеров
может иметь сильный сигнал, но почти параллельные строки; крайние размеры могут
лучше разделять слои, но реальный малый размер может не пройти порог
измеримости. Число обусловленности также не заменяет абсолютную амплитуду и
ковариацию.

Для модуля $|Z^*|$ нужен подтверждённый оператор наблюдения. Реальная
ковариация двух последовательных записей должна передаваться явно после
выполнения `31.04`.


In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
model_paths = sorted({
    (candidate / "two_layer_model.py").resolve()
    for candidate in candidates
    if (candidate / "two_layer_model.py").is_file()
})
if len(model_paths) != 1:
    raise RuntimeError(f"expected one canonical two_layer_model.py, found: {model_paths}")

model_path = model_paths[0]
sys.path.insert(0, str(model_path.parent))
import two_layer_model as tlm

if Path(tlm.__file__).resolve() != model_path:
    raise RuntimeError(f"imported non-canonical model: {tlm.__file__}")

print(f"Python {platform.python_version()}; NumPy {np.__version__}")
print(f"Модель: {model_path}")


In [ ]:
# Синтетическая рабочая точка; не параметры добровольца.
RHO1 = 5.0
RHO2 = 20.0
RHO2_EXHALE = 15.0
RHO2_INHALE = 25.0
H = 0.020
BETA = 0.5
MODEL_VALIDITY = "unverified_without_subject_specific_CT_FEM_Lmax"
OBSERVATION_OPERATOR = "signed_real_Z; experimental_magnitude_not_yet_verified"

SIZE_SETS_MM = {
    "9 размеров; поздняя копия 100 мм исключена": (50, 60, 70, 80, 90, 110, 120, 130, 140),
    "10 размеров; 90 и 100 мм независимы до QC": (50, 60, 70, 80, 90, 100, 110, 120, 130, 140),
}

print("status=illustrative_unvalidated")
print(f"model_validity={MODEL_VALIDITY}")
print(f"observation_operator={OBSERVATION_OPERATOR}")
print(f"rho1={RHO1:g} Ом·м; rho2={RHO2:g} Ом·м; h={H*1000:g} мм; beta={BETA:g}")
for name, sizes in SIZE_SETS_MM.items():
    print(f"{name}: {sizes}")


In [ ]:
def log_jacobian(sizes_m, rho1=RHO1, rho2=RHO2, h=H):
    rows = []
    for size_m in np.asarray(sizes_m, dtype=float):
        a, b = tlm.geometry_from_size(float(size_m), BETA)
        result = tlm.evaluate(rho1, rho2, h, a, b)
        rows.append([
            rho1 * result.d_rho1 / result.z,
            rho2 * result.d_rho2 / result.z,
        ])
    matrix = np.asarray(rows, dtype=float)
    if not np.isfinite(matrix).all():
        raise RuntimeError("non-finite logarithmic Jacobian")
    return matrix


def breathing_delta(size_m, h=H):
    a, b = tlm.geometry_from_size(float(size_m), BETA)
    return (
        tlm.transfer_impedance(RHO1, RHO2_INHALE, h, a, b)
        - tlm.transfer_impedance(RHO1, RHO2_EXHALE, h, a, b)
    )


def conditional_covariance(log_jacobian_matrix, relative_error_covariance):
    jacobian = np.asarray(log_jacobian_matrix, dtype=float)
    covariance = np.asarray(relative_error_covariance, dtype=float)
    if covariance.shape != (jacobian.shape[0], jacobian.shape[0]):
        raise ValueError("covariance shape must match observations")
    if not np.allclose(covariance, covariance.T, rtol=0.0, atol=1e-12):
        raise ValueError("covariance must be symmetric")
    cholesky = np.linalg.cholesky(covariance)
    whitened = np.linalg.solve(cholesky, jacobian)
    singular = np.linalg.svd(whitened, compute_uv=False)
    if singular[-1] <= np.finfo(float).eps * singular[0]:
        raise np.linalg.LinAlgError("whitened Jacobian is rank deficient")
    return np.linalg.inv(whitened.T @ whitened)


def pair_metrics(size_i_m, size_j_m):
    matrix = log_jacobian([size_i_m, size_j_m])
    singular = np.linalg.svd(matrix, compute_uv=False)
    row_norms = np.linalg.norm(matrix, axis=1)
    cosine = np.dot(matrix[0], matrix[1]) / np.prod(row_norms)
    unit_covariance = conditional_covariance(matrix, np.eye(2))
    return {
        "Li_mm": int(round(size_i_m * 1000)),
        "Lj_mm": int(round(size_j_m * 1000)),
        "sigma_min": float(singular[-1]),
        "condition": float(singular[0] / singular[-1]),
        "angle_deg": float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))),
        "unit_var_log_rho2": float(unit_covariance[1, 1]),
        "min_abs_delta_z": float(min(abs(breathing_delta(size_i_m)), abs(breathing_delta(size_j_m)))),
    }


def enumerate_pairs(sizes_mm):
    sizes_m = np.asarray(sizes_mm, dtype=float) / 1000.0
    table = []
    for index, size_i in enumerate(sizes_m):
        for size_j in sizes_m[index + 1:]:
            table.append(pair_metrics(float(size_i), float(size_j)))
    return table


def pareto_front(table):
    # Максимизируются sigma_min и min_abs_delta_z; веса намеренно не задаются.
    front = []
    for candidate in table:
        dominated = any(
            other["sigma_min"] >= candidate["sigma_min"]
            and other["min_abs_delta_z"] >= candidate["min_abs_delta_z"]
            and (
                other["sigma_min"] > candidate["sigma_min"]
                or other["min_abs_delta_z"] > candidate["min_abs_delta_z"]
            )
            for other in table
        )
        if not dominated:
            front.append(candidate)
    return sorted(front, key=lambda row: row["Li_mm"])


# Самопроверки: одинаковые размеры вырождены; 50/140 имеет полный локальный ранг.
assert np.linalg.matrix_rank(log_jacobian([0.050, 0.050])) == 1
assert np.linalg.matrix_rank(log_jacobian([0.050, 0.140])) == 2
np.testing.assert_allclose(log_jacobian([0.050, 0.140]).sum(axis=1), 1.0, rtol=1e-10, atol=1e-12)
assert len(enumerate_pairs(SIZE_SETS_MM[next(iter(SIZE_SETS_MM))])) == 36
assert len(enumerate_pairs(SIZE_SETS_MM[list(SIZE_SETS_MM)[1]])) == 45
print("Самопроверки перебора, ранга и условной ковариации: пройдены")


In [ ]:
all_tables = {name: enumerate_pairs(sizes) for name, sizes in SIZE_SETS_MM.items()}

for set_name, table in all_tables.items():
    print(f"\n{set_name}; формальных пар: {len(table)}")
    rankings = {
        "max sigma_min (структурное разделение)": sorted(table, key=lambda row: row["sigma_min"], reverse=True),
        "min unit_var_log_rho2 (Sigma=I)": sorted(table, key=lambda row: row["unit_var_log_rho2"]),
        "max min|DeltaZ| (только амплитуда)": sorted(table, key=lambda row: row["min_abs_delta_z"], reverse=True),
    }
    for criterion, ordered in rankings.items():
        print(f"  {criterion}:")
        for row in ordered[:3]:
            print(
                f"    {row['Li_mm']:3d}-{row['Lj_mm']:3d} мм; "
                f"sigma_min={row['sigma_min']:.5f}; cond={row['condition']:.2f}; "
                f"unit_var_ln_rho2={row['unit_var_log_rho2']:.2f}; "
                f"min|DeltaZ|={row['min_abs_delta_z']:.3f} Ом"
            )

    front = pareto_front(table)
    print("  Pareto-front [sigma_min, min|DeltaZ|], без весов:")
    for row in front:
        print(
            f"    {row['Li_mm']:3d}-{row['Lj_mm']:3d} мм: "
            f"sigma_min={row['sigma_min']:.5f}; min|DeltaZ|={row['min_abs_delta_z']:.3f} Ом"
        )


In [ ]:
def solve_positive_signed_pair(pair_sizes_m, pair_z, h_m, initial=(5.0, 20.0), max_iter=30):
    pair_sizes_m = np.asarray(pair_sizes_m, dtype=float)
    observations = np.asarray(pair_z, dtype=float)
    if pair_sizes_m.shape != (2,) or observations.shape != (2,) or np.any(observations <= 0):
        raise ValueError("this solver requires two positive signed-Z observations")

    log_parameters = np.log(np.asarray(initial, dtype=float))
    for _ in range(max_iter):
        rho1, rho2 = np.exp(log_parameters)
        predictions = []
        for size_m in pair_sizes_m:
            a, b = tlm.geometry_from_size(float(size_m), BETA)
            predictions.append(tlm.transfer_impedance(rho1, rho2, h_m, a, b))
        predictions = np.asarray(predictions)
        residual = np.log(predictions) - np.log(observations)
        if np.linalg.norm(residual, ord=np.inf) < 1e-12:
            return rho1, rho2
        step = np.linalg.solve(log_jacobian(pair_sizes_m, rho1, rho2, h_m), residual)
        log_parameters -= step
    raise RuntimeError("pair Newton solver did not converge")


# Только algebra/code self-test: те же синтетические модель и данные не валидируют метод.
true_parameters = (6.0, 18.0)
test_pair = np.array([0.050, 0.140])
test_observations = []
for size_m in test_pair:
    a, b = tlm.geometry_from_size(float(size_m), BETA)
    test_observations.append(tlm.transfer_impedance(*true_parameters, H, a, b))
recovered = solve_positive_signed_pair(test_pair, test_observations, H)
np.testing.assert_allclose(recovered, true_parameters, rtol=1e-10, atol=1e-10)

held_out_sizes = np.array([0.060, 0.090, 0.120])
held_out_residuals = []
for size_m in held_out_sizes:
    a, b = tlm.geometry_from_size(float(size_m), BETA)
    truth = tlm.transfer_impedance(*true_parameters, H, a, b)
    prediction = tlm.transfer_impedance(*recovered, H, a, b)
    held_out_residuals.append(prediction - truth)
np.testing.assert_allclose(held_out_residuals, 0.0, rtol=0.0, atol=1e-10)
print("Синтетический self-test обращения пары и holdout-предсказания: пройден")


## Интерпретация и запрет преждевременного выбора

В синтетической точке невзвешенный структурный критерий обычно предпочитает
далеко разнесённые размеры, тогда как критерий только по минимальной амплитуде
тяготеет к двум крупным размерам. Последние имеют близкие строки
чувствительности и потому хуже разделяют слои. Это демонстрирует конфликт
целей, но не выбирает одну из них за исследователя.

Pareto-front сохраняет пары, которые нельзя одновременно улучшить по
$\sigma_{min}$ и синтетическому минимуму $|\Delta Z|$. Он также не является
готовым ответом: изменение рабочей точки, ковариации, оператора наблюдения,
$h$, допустимого $L_{max}$ или экспериментальной повторяемости изменит front.

Синтетический Newton/holdout-тест проверяет только согласованность кода с той же
моделью, которая породила данные. Нулевой остаток здесь не является внешней
валидацией. В реальном анализе пара имеет два наблюдения на два параметра и не
оставляет степеней свободы; адекватность проверяется всеми не вошедшими в пару
размерами.


## Условия реального выбора пары

Для каждого добровольца отдельно необходимо:

1. принять дыхательную разметку и QC всех записей с хешами;
2. подставить $h_{CT}$ с правилом получения и неопределённостью;
3. подтвердить для реокардиомонитора МГТУ оператор наблюдения, gain/offset и
   шкалу переменного канала;
4. оценить ковариацию последовательных записей по реестру `31.04`, не
   подставляя остаток подгонки как независимый шум;
5. исключить 100 мм только в субъектном наборе с поздней копией; для второго
   добровольца решить судьбу 90/100 мм по QC их собственного канала;
6. отфильтровать размеры по индивидуальной применимости КТ/FEM (`20.12`);
7. для каждой пары решить нелинейную задачу, предсказать все holdout-размеры,
   выполнить leave-one-size-out/устойчивость к $h$ и проверить границы;
8. не использовать одну и ту же запись одновременно для выбора пары и
   оптимистической оценки её точности без bootstrap/независимого блока.

Текущий итог:

```text
pair_status = candidate_not_selected
reason = calibration_covariance_CT_h_model_validity_and_holdout_missing
```

`32.03` рассматривает сокращённую постановку с неизвестной $h$ и не должен
считать результат этой полной постановки автоматически переносимым.
